In [84]:
pip install pytest-playwright

Note: you may need to restart the kernel to use updated packages.


In [85]:
!playwright install

In [1]:
from playwright.async_api import async_playwright
import time
import asyncio
import pandas as pd
import re

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless = False, args=["--start-maximized"])
page = await browser.new_page()
await page.set_viewport_size({"width": 1920, "height": 1080})

url = "https://app.folk.app/shared/US-VCs-oc71Oi94yB9vwbfh1XWIQPHTAGQE7FQ1"
await page.goto(url)
time.sleep(3)

titles = ["Person"] +\
    (await page.locator("#root div[data-testid='table-virtualizer'] > div > div:nth-child(2) > div").inner_text()).split("\n")

total_list = []
current_last = 0
# TODO(Henry): Add try-catch blocks for each attempt, leave "Error" data for failures
while True:
    name_list = await page.locator("#root div[data-testid='table-virtualizer'] > div > div:nth-child(3) > div").all()
    
    # Click the first loaded to hack scroll to the left
    # next() is not usable here due to a bug in async generator
    valid_first = None
    for x in name_list:
        if int(await x.get_attribute("aria-rowindex")) > current_last:
            valid_first = x
            break
    if valid_first is not None:
        await valid_first.click()
        time.sleep(0.3)
    row_list = await page.locator("#root div[role='row']").all()
    
    # Find the last loaded row
    for last in reversed(row_list):
        if await last.locator(">div").first.inner_html():
            new_last = int(await last.get_attribute("aria-rowindex"))
            break
    # Buttom reached
    if current_last == new_last:
        break
    
    # Souping rows
    name_content = [
        (await x.inner_text()).split("\n")[-1:] for x in name_list
        if int(await x.get_attribute("aria-rowindex")) > current_last
        and int(await x.get_attribute("aria-rowindex")) <= new_last
    ]
    row_element = [
        x for x in row_list
        if int(await x.get_attribute("aria-rowindex")) > current_last
        and int(await x.get_attribute("aria-rowindex")) <= new_last
    ]
    row_content = []
    
    # Souping cols
    current_col = 1
    while True:
        # Find the last loaded col
        new_col = int(await (await row_element[0].locator(">div>div:nth-child(1)").all())[-1].get_attribute("aria-colindex"))
        col_element = []
        for x in row_element:
            buf_col = []
            buf_items = await x.locator(">div").all()
            for y in buf_items:
                col_index = int(await y.locator(">div:nth-child(1)").get_attribute("aria-colindex"))
                if col_index > current_col and col_index <= new_col:
                    buf_col.append(y)
            col_element.append(buf_col)
        col_content = []
        # This line could work without importing asyncio (a little slower),
        # but a bug in python<3.11 reports it as syntax error. Fixed in python 3.11
#         col_content = [[await element.inner_text() for element in elements] for elements in col_element]
        col_content = [await asyncio.gather(*[element.inner_text() for element in elements]) for elements in col_element]
        if row_content:
            row_content = [x + y for x, y in zip(row_content, col_content)]
        else:
            row_content = col_content
        if current_col == new_col:
            break
        current_col = new_col
        await col_element[0][-1].scroll_into_view_if_needed()
        time.sleep(0.2)
    total_list += [x + y for x, y in zip(name_content, row_content)]
    current_last = new_last
    # Jump back to clicked element (to scroll left)
    await page.keyboard.press("ArrowLeft")
    time.sleep(0.3)
    await row_list[-1].scroll_into_view_if_needed()
    time.sleep(0.3)



title_list = []
result_df = pd.DataFrame(total_list, columns=titles)
result_df["Companies"] = result_df["Companies"].apply(lambda x: re.sub(r'^.*?\n', '', x))
result_df = result_df.replace({'\n': ';'}, regex=True)
result_df.to_csv("./folkapp1.csv")

await browser.close()
await playwright.stop()
playwright = None
browser = None
page = None


C:\Users\why\AppData\Local\Temp\ipykernel_22380\2442751619.py:103: RuntimeWarning: coroutine 'Browser.close' was never awaited
  browser.close()


<coroutine object PlaywrightContextManager.__aexit__ at 0x0000000000B0B240>

In [2]:
result_df

,Person,Emails,Urls,Companies,Portfolio companies,Fund type,Fund stage,Fund focus,Location,Twitter Link,LinkedIn Link,Facebook Link,Number of Investments,Number of Exits,Fund Description,Founding Year,Description
0,Susan Akbarpour,susan@candouventures.com,https://www.linkedin.com/in/susanakbarpour,Candou Ventures,SignalRank Corporation,Venture Fund,Seed,Finance & Crypto;Software & Internet,Palo Alto,,https://www.linkedin.com/company/candou-ventures/,,9,0,Candou is an early seed round firm focused on ...,2015,"Investor, Serial entrepreneur and board member"
1,Soham Avlani,soham@9unicorns.in,https://www.linkedin.com/in/savlani,9Unicorns,"GOQii, Vested Finance",Accelerator,Series A;Series B;Seed,Finance & Crypto;Consumer;Health,Mumbai;India,https://twitter.com/9UnicornsVC,https://www.linkedin.com/company/9unicorns/about/,https://www.facebook.com/9UnicornsVC,155,1,9Unicorns is a stage and sector agnostic accel...,2020,Partner - 9Unicorns Fund | Venture Catalysts |...
2,Vasudev Bailey,vb@av.co,https://www.linkedin.com/in/baileyv,Artis Ventures (AV),"YouTube, Lemonaid Health, Activ Surgical, Tast...",Venture Fund,Seed;Pre-Seed;Series A;Series B;Series C;Series D,Health;Entertainment & Media;AI & Machine Lear...,San Francisco;California,http://twitter.com/artisventures,http://www.linkedin.com/company/artis-ventures,http://www.facebook.com/pages/ARTIS-Ventures/3...,101,27,ARTIS Ventures is a financial services firm th...,2001,
3,Victoria Beasley Vbeasle,vbeasley@preludeventures.com,www.preludeventures.com,Prelude Ventures,"LevelTen Energy, Voltus, Scoop Technologies, T...",Venture Fund,Seed;Pre-Seed;Series A;Series B;Series C;Series D,Software & Internet;AI & Machine Learning;Logi...,San Francisco;California,http://twitter.com/PreludeVC,http://www.linkedin.com/company/prelude-ventures,http://www.facebook.com/preludeventures,109,10,Prelude invests in market moving startups addr...,2013,
4,Richard Brekka Brekka,rbrekka@secondalpha.com,http://secondalpha.com/,Second Alpha Partners,,Venture Fund,Series D;Series C,Finance & Crypto;AI & Machine Learning;Marketi...,New York,http://twitter.com/secondalphanews,http://www.linkedin.com/company/second-alpha,http://www.facebook.com/pages/Second-Alpha/383...,5,2,Second Alpha Partners is a New York based priv...,2012,Founder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2211,Mike Gregoire,mike@brightonparkcap.com,www.brightonparkcap.com,Brighton Park Capital,Reltio,Venture Fund,Seed;Pre-Seed;Series A,Software & Internet,Greenwich;Connecticut,,https://www.linkedin.com/company/brighton-park...,,16,3,Brighton Park Capital is an investment firm th...,2019,
2212,Robert Schwartz,rob@thirdpointventures.com,http://www.thirdpointventures.com/,Third Point Ventures,"Balbix, Trullion, Blameless, Ahana, Ushur, Packet",Venture Fund,Seed;Series A;Series B;Series C;Series D,Software & Internet;AI & Machine Learning;Fina...,New York,,https://www.linkedin.com/company/71879,,96,26,Third Point Ventures is an investment firm tha...,1995,
2213,Kip McClanahan,kip@silvertonpartners.com,www.silvertonpartners.com,Silverton Partners,"Bennie, Billie, Self Financial, SourceDay, Cle...",Venture Fund,Seed;Pre-Seed;Series A,Software & Internet;Health;AI & Machine Learni...,Austin;Texas,http://twitter.com/silvertonvc,https://www.linkedin.com/company/1026196,https://www.facebook.com/silvertonpartners,184,37,Silverton Partners is an early-stage venture c...,2006,
2214,Marten Vading,marten@kreoscapital.com,http://www.kreoscapital.com,Kreos Capital,"Dreamlines, Quali",Venture Fund,Seed;Series A;Series B;Series C;Series D,Software & Internet;Logistics;Transportation;C...,London;United Kingdom,https://twitter.com/kreoscapital,https://www.linkedin.com/company/kreos-capital,https://www.facebook.com/Kreos-Capital-8165020...,120,46,Kreos Capital is a growth debt firm that provi...,1998,
